# 10 — Installation check

Check the install from `08_getting_started.md` works, and run one op.

## Did our installation work?

Run this notebook in the environment you installed in 08.  Does it let you know where sci-kit ops and napari-ai-lab exist?

In [1]:
import skop
import napari_ai_lab

print('skop         ', skop.__file__)
print('napari_ai_lab', napari_ai_lab.__file__)

skop          C:\Users\bnort\work\ImageJ2022\tnia\i2k-2026\pixi\.pixi\envs\default\Lib\site-packages\skop\__init__.py
napari_ai_lab C:\Users\bnort\work\ImageJ2022\tnia\i2k-2026\pixi\.pixi\envs\default\Lib\site-packages\napari_ai_lab\__init__.py


## Why micro_sam is in this environment

- We need it for interactive labelling
- SAM creates 'embeddings' which are medium expensive (20-60 seconds)
- For interactive labelling we want to keep the embedding in memory
  - First interactive label takes 20-60 seconds
  - The rest are a fast (a second or so)
- Currently microsam needs to be in host environment so the embeddings are available
- In the future ops should keep some artifacts (models, embeddings, histograms) persistent between calls.
  
**Isolation is for dependencies that are incompatible, not dependencies
that are complicated.**

For example

- Cellpose 3 and Cellpose 4 cannot be installed together. Either one could be in the host, but one has to be in an isolated environment. 
- StarDist needs TensorFlow, which will is difficult to install with a recent PyTorch (especially on Windows where old versions of Tensorflow are needed for GPU support)

In [2]:
import micro_sam, torch
from micro_sam.multi_dimensional_segmentation import segment_mask_in_volume
from micro_sam.sam_annotator._state import AnnotatorState
from micro_sam.sam_annotator.util import prompt_segmentation

print('micro_sam', micro_sam.__version__)
print('torch    ', torch.__version__, '| CUDA:', torch.cuda.is_available())
print("SAM3D's entry points import OK")

micro_sam 1.8.14
torch     2.10.0 | CUDA: True
SAM3D's entry points import OK


## Run an op and get it to report back

### What is an op

An op is a function that expresses information about how to run it with decorators and annotations.  See an example [here](https://github.com/apposed/scikit-ops/blob/main/src/skop/ops/segment/stardist2d.py#L36)

### What is a runner? 

Instead of 

```
func(a,b,c)
```

We do this

```
runner.run(func, a, b, c)
```

A runner decides how a function is run.  It can pre-process the input (for example slice a ND image, or tile a large one), run in an isolated environment or an HPC, handle progress, and deal with crashes. 

Below the op `slow_sum` runs both explicity and by the runner in the light `minimal` env and emits four progress events.

In [4]:
import numpy as np
from skop.runner import Runner
from skop.ops.toy import slow_sum

# run slow sum directory
print('run slow_sum directly')
result = slow_sum(np.ones((4, 4), np.float32), steps=4)
print('result of direct slow_sum:', result)
print()

# run slow sum using the runner
runner = Runner()

def show(ev):
    print('EVENT', ev.current, ev.maximum, ev.message, flush=True)

total = runner.run(slow_sum, image=np.ones((4, 4), np.float32), steps=4,
                   on_progress=show)
print('result:', total)

run slow_sum directly
result of direct slow_sum: 16.0

EVENT None None None
EVENT 0 4 Summing chunk 1 of 4
EVENT 1 4 Summing chunk 2 of 4
EVENT 2 4 Summing chunk 3 of 4
EVENT 3 4 Summing chunk 4 of 4
EVENT None None None
result: 16.0


## Next

Go to notebook 12 to get the data, then notebook 15 to build the environments.